# Dataset pipeline, stages 1 and 2

Stage 1: find the big move days from price data.
Stage 2: pull the news documents for those days.

Run this on Colab, it needs internet for both the price data and the news.

## Setup

In [ ]:
!pip install -q yfinance trafilatura
# big_moves.py and retrieval.py need to be in the same folder.
# If running fresh on Colab, upload them or clone the repo first.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from big_moves import (select_big_move_days, standardised_returns,
                       lee_mykland, log_returns)
from retrieval import TICKERS, fetch_prices, build_topic

pd.set_option("display.width", 120)
print("ready")

## Stage 1: price data and big move days

Four assets. Start date is set well before the period we want so the
volatility estimates have enough history to warm up.

In [ ]:
START = "2019-01-01"
END   = "2024-12-31"

prices = {}
for asset, ticker in TICKERS.items():
    try:
        prices[asset] = fetch_prices(ticker, START, END)
        print(f"{asset:8s} {ticker:10s} {len(prices[asset]):5d} days  "
              f"{prices[asset].index[0].date()} to {prices[asset].index[-1].date()}")
    except Exception as e:
        print(f"{asset:8s} FAILED: {e}")

### How different are the four assets?

This is the reason a fixed percentage threshold would not work. Each asset
has a very different idea of what a normal day looks like.

In [ ]:
rows = []
for asset, p in prices.items():
    r = log_returns(p).dropna()
    rows.append({
        "asset": asset,
        "daily vol %": r.std()*100,
        "annualised vol %": r.std()*np.sqrt(252)*100,
        "largest 1d move %": (np.exp(r.abs().max())-1)*100,
        "days >2% move": int((r.abs() > 0.02).sum()),
    })
pd.DataFrame(rows).round(2)

### Detecting the big move days

Both methods, so we can compare. The z-score version is simple and flags
more days. Lee-Mykland is the published test, uses bipower variation so a
jump cannot inflate its own volatility estimate, and sets its threshold
from the Gumbel distribution rather than by hand.

In [ ]:
results = {}
for asset, p in prices.items():
    z_days  = select_big_move_days(p, method="zscore", z_threshold=3.0)
    lm_days = select_big_move_days(p, method="lee_mykland", alpha=0.01)
    results[asset] = {"zscore": z_days, "lee_mykland": lm_days}
    print(f"{asset:8s} z-score flagged {len(z_days):3d}   "
          f"lee-mykland flagged {len(lm_days):3d}")

### Sanity check

If the detector works, the days it flags should line up with events we
already know about. Look for late Feb and early Mar 2022 (Ukraine), Nov
2022 (FTX), Mar 2023 (Silicon Valley Bank), Sep 2022 (the mini-budget) and
Mar 2020 (covid).

In [ ]:
for asset in results:
    print(f"\n=== {asset.upper()} : top 10 by Lee-Mykland ===")
    df = results[asset]["lee_mykland"].head(10)
    show = df[["pct_move", "direction", "score"]].copy()
    show.index = show.index.date
    print(show.round(2).to_string())

In [ ]:
# Visual check: flagged days marked on the price series
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, (asset, p) in zip(axes.flat, prices.items()):
    ax.plot(p.index, p.values, linewidth=0.8, color="#4C72B0")
    flagged = results[asset]["lee_mykland"]
    ax.scatter(flagged.index, p.reindex(flagged.index).values,
               color="red", s=18, zorder=5, label=f"{len(flagged)} flagged")
    ax.set_title(f"{asset} ({TICKERS[asset]})")
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

### Choosing how many events to keep

Lee-Mykland gives a statistically defined set, but the count will differ by
asset. For a balanced dataset we take the top N from each asset by test
statistic, so no single asset dominates the way a few topics dominate
AER.

In [ ]:
N_PER_ASSET = 25

selected = []
for asset in results:
    top = results[asset]["lee_mykland"].head(N_PER_ASSET)
    for date, row in top.iterrows():
        selected.append({
            "asset": asset,
            "date": date.strftime("%Y-%m-%d"),
            "pct_move": round(row["pct_move"], 2),
            "direction": row["direction"],
            "score": round(row["score"], 2),
        })

selected = pd.DataFrame(selected).sort_values(["asset", "date"])
print(f"{len(selected)} candidate events, {selected.groupby('asset').size().to_dict()}")
selected.head(20)

In [ ]:
selected.to_csv("big_move_days.csv", index=False)
print("saved big_move_days.csv")

## Stage 2: news retrieval

For each event, pull articles from a window around that date. The window
leans backwards because causes precede the move. A small forward window
catches same-story reporting that landed late, but it is kept short, since
anything well after the move is a consequence rather than a cause.

Distractor documents are pulled with deliberately off-chain queries. Without
them the retrieval part of the task is too easy, which is what AER did.

In [ ]:
# Start with a couple of events to check the retrieval works before
# running the whole set. Fetching article text is slow and some URLs fail.
test_events = selected.head(2).to_dict("records")

topics = []
for ev in test_events:
    print(f"\nfetching for {ev['asset']} on {ev['date']} ({ev['pct_move']}%)")
    topic = build_topic(ev["asset"], ev["date"],
                        n_relevant=15, n_distractor=5, fetch_text=True)
    topics.append(topic)
    print(f"  {topic['n_with_text']} of {topic['n_requested']} documents had usable text")

### Did we get enough evidence?

AER averages 19.7 documents per topic and about 28,000 tokens of evidence.
That volume is a large part of what makes the task hard, because the model
has to find the relevant events inside it. If we come out far below that,
the retrieval challenge disappears and the questions become too easy.

In [ ]:
for t in topics:
    words = sum(len((d.get("content") or "").split()) for d in t["docs"])
    n_rel = sum(1 for d in t["docs"] if d["role"]=="relevant")
    n_dis = sum(1 for d in t["docs"] if d["role"]=="distractor")
    print(f"{t['asset']} {t['event_date']}: {len(t['docs'])} docs "
          f"({n_rel} relevant, {n_dis} distractor), ~{words} words, "
          f"~{int(words*1.3)} tokens")
print("\nAER reference: 19.7 docs, ~28,000 tokens per topic")

In [ ]:
sources = {}
for t in topics:
    for d in t["docs"]:
        sources[d["source"]] = sources.get(d["source"], 0) + 1
print(f"{len(sources)} unique sources so far")
print(sorted(sources.items(), key=lambda x: -x[1])[:15])

### Run the rest

Only do this once the checks above look reasonable. It takes a while
because of the per-article delay.

In [ ]:
from checkpoint import resumable_map, checkpoint_status

RUN_ALL = False   # flip to True when ready

if RUN_ALL:
    events = selected.to_dict("records")

    def fetch(ev):
        t = build_topic(ev["asset"], ev["date"])
        t["asset"] = ev["asset"]
        t["event_date"] = ev["date"]
        t["target_event"] = f"{ev['asset']} moved {ev['pct_move']}% on {ev['date']}"
        return t

    topics = resumable_map(
        items   = events,
        key_fn  = lambda ev: f"{ev['asset']}_{ev['date']}",
        work_fn = fetch,
        path    = "topics_progress.json",
    )

    # topic_id has to be assigned after, so it stays stable across resumes
    for i, t in enumerate(topics):
        t["topic_id"] = i

    import json
    with open("topics.json", "w") as f:
        json.dump(topics, f, indent=2)
    print(f"\nsaved {len(topics)} topics to topics.json")

In [ ]:
# If the session dropped, run this to see where you got to,
# then just re-run the cell above. It picks up from the checkpoint.
print(checkpoint_status("topics_progress.json"))

## What comes next

Stage 3 extracts candidate cause events from these documents and puts them
on a timeline. Stage 4 scores each candidate with three models and routes
the ones they disagree on to a human. Neither is written yet, and stage 4
depends on the annotation decisions that are still open.